# Birthday Probe Explorer

Interactive companion to `Training_On_LM4/eval/birthday_probe_legacy.py`.

Loads a trained Llama checkpoint + its dataset artifacts and lets you:

- pick a **person** from the sampled population,
- run the 4 birthday-probe metrics (**MP / Day|M / Year|M,D / FP**) on any
  *(person, template)* pair, picking which metric to focus on,
- see *what the model predicted instead* of the truth — per-position top-k and a
  full probability distribution over the 12 months,
- sweep every template for one person to find which phrasings work / fail,
- type **free-form prompt prefixes** and watch the model decode.

The probe logic (`build_chunks`, teacher-forced argmax positions, greedy FP decode)
is copied verbatim from `birthday_probe_legacy.py`, so numbers here match the CLI probe.

## 1. Setup

In [17]:
import sys, json, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, GPT2Tokenizer

# Locate the repo root (works whether the kernel started in Interp_LM4/ or the repo root).
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "Training_On_LM4").is_dir():
            return p
    raise RuntimeError(f"Could not find 'Training_On_LM4' above {start}")

REPO_ROOT     = find_repo_root(Path.cwd().resolve())
TRAINING_ROOT = REPO_ROOT / "Training_On_LM4"
if str(TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINING_ROOT))   # so `import data.bio_text` resolves

from data.bio_text import FIELD_SPECS
from data.sample_people import sample_people, MONTHS

print("repo root     :", REPO_ROOT)
print("training root :", TRAINING_ROOT)

repo root     : /Users/efmac/Code/Project Code/CRL-Interp
training root : /Users/efmac/Code/Project Code/CRL-Interp/Training_On_LM4


## 2. Point at a model + dataset

Edit these two paths to probe a different checkpoint. The defaults are the
`BD_llama_inital` data and the `BD_llama_6heads_1epoch_4layers` model.

In [18]:
# === EDIT HERE =============================================================
MODEL_DIR = REPO_ROOT / "Interp_LM4" / "model" / "BD_llama_6heads_1epoch_4layers"
DATA_DIR  = REPO_ROOT / "Interp_LM4" / "data"  / "BD_llama_inital"
# ===========================================================================

assert MODEL_DIR.is_dir(), f"missing model dir: {MODEL_DIR}"
assert DATA_DIR.is_dir(),  f"missing data dir:  {DATA_DIR}"
for f in ("config.json", "model.safetensors"):
    assert (MODEL_DIR / f).exists(), f"missing {f} in {MODEL_DIR}"

print("model :", MODEL_DIR.name)
print("data  :", DATA_DIR.name)
print("data files :", sorted(p.name for p in DATA_DIR.iterdir()))

model : BD_llama_6heads_1epoch_4layers
data  : BD_llama_inital
data files : ['bios_postreduce.bin', 'bios_prereduce.bin', 'data_config.json', 'old_to_new.json', 'people.json']


## 3. Rebuild the dataset artifacts

`data_config.json` records the exact `(N, seed)` used at training time. We load the
saved `people.json` if present (else resample identically), and load the GPT-2 ->
reduced-vocab remap the model was trained with (`old_to_new.json`, else rebuilt from
`bios_prereduce.bin`).

In [19]:
data_cfg = json.loads((DATA_DIR / "data_config.json").read_text())
print("data_config.json:")
for k in ("NAME", "N", "K", "SEED", "SHUFFLE_SEED", "FIELDS",
          "reducedVocabSize", "reducedEOSToken", "EPOCHS"):
    if k in data_cfg:
        print(f"  {k:18s}= {data_cfg[k]}")

N    = data_cfg["N"]
SEED = data_cfg["SEED"]

# --- people: prefer the saved people.json, else resample deterministically ---
people_json = DATA_DIR / "people.json"
if people_json.exists():
    people = json.loads(people_json.read_text())
    print(f"\nLoaded {len(people):,} people from people.json")
else:
    people = sample_people(N=N, seed=SEED)
    print(f"\nResampled {len(people):,} people  (sample_people N={N}, seed={SEED})")

# --- vocab remap: prefer saved old_to_new.json, else rebuild from the token file ---
o2n_json = DATA_DIR / "old_to_new.json"
if o2n_json.exists():
    old_to_new = {int(k): int(v) for k, v in json.loads(o2n_json.read_text()).items()}
    print(f"Loaded vocab remap from old_to_new.json  ({len(old_to_new):,} entries)")
else:
    from data.tokenize_pack import build_vocab_remap
    old_to_new, _, _ = build_vocab_remap(str(DATA_DIR / "bios_prereduce.bin"))
    print(f"Rebuilt vocab remap from bios_prereduce.bin  ({len(old_to_new):,} entries)")

new_to_old    = {v: k for k, v in old_to_new.items()}
reduced_vocab = len(old_to_new)
templates     = FIELD_SPECS["birthday"]["templates"]
print(f"reduced vocab : {reduced_vocab}")
print(f"templates     : {len(templates)} birthday paraphrases")

data_config.json:
  NAME              = bioS_name_date_small_lama_abalation
  N                 = 50000
  K                 = 100
  SEED              = 0
  SHUFFLE_SEED      = 1
  FIELDS            = ['birthday']
  reducedVocabSize  = 1836
  reducedEOSToken   = 1835
  EPOCHS            = 1.0

Loaded 50,000 people from people.json
Loaded vocab remap from old_to_new.json  (1,836 entries)
reduced vocab : 1836
templates     : 46 birthday paraphrases


## 4. Load the model

In [20]:
def pick_device() -> str:
    if torch.cuda.is_available():         return "cuda"
    if torch.backends.mps.is_available(): return "mps"
    return "cpu"

DEVICE = pick_device()
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

eos_remapped = old_to_new[int(tokenizer.eos_token_id)]

print(f"device          : {DEVICE}")
print(f"model vocab     : {model.config.vocab_size}")
print(f"layers / heads  : {model.config.num_hidden_layers} / {model.config.num_attention_heads}")
print(f"hidden size     : {model.config.hidden_size}")
print(f"eos (remapped)  : {eos_remapped}")
if model.config.vocab_size != reduced_vocab:
    print(f"  !! WARNING: model vocab {model.config.vocab_size} != remap {reduced_vocab}")

Loading weights: 100%|██████████| 38/38 [00:00<00:00, 8717.58it/s]

device          : mps
model vocab     : 1836
layers / heads  : 4 / 6
hidden size     : 384
eos (remapped)  : 1835


## 5. Probe core

Verbatim from `birthday_probe_legacy.py` (`build_chunks`, the teacher-forced argmax
positions, the greedy FP decode), plus a *detailed* scorer that also records
per-position top-k predictions so we can see **how** the model fails, not just that
it failed.

In [21]:
def build_chunks(person: dict, template: str) -> dict:
    """Split a birthday template into prefix / month / day / sep / year / trailing.

    Verbatim from birthday_probe_legacy.build_chunks. Concatenating
    prefix+month+day+sep+year+trailing reproduces the bio text seen at training.
    """
    name = f"{person['first_name']} {person['middle_name']} {person['last_name']}"
    before, after = template.split("{birthday}", 1)
    return {
        "prefix":   (" " + before.format(name=name)).rstrip(" "),
        "month":    f" {person['birthmonth']}",
        "day":      f" {person['birthday']}",
        "sep":      ",",
        "year":     f" {person['birthyear']}",
        "trailing": after,
    }


class MissingTokenError(KeyError):
    """Raised when a token is not in the model's reduced vocab."""


def tokenize_and_remap(text: str, *, strict: bool = True) -> list[int]:
    """GPT-2 tokenize `text` then remap ids into the model's reduced vocab.

    strict=True  -> raise MissingTokenError on any unseen token.
    strict=False -> silently drop unseen tokens (for free-form exploration).
    """
    raw = tokenizer(text, add_special_tokens=False)["input_ids"]
    out = []
    for t in raw:
        t = int(t)
        if t in old_to_new:
            out.append(old_to_new[t])
        elif strict:
            raise MissingTokenError(
                f"token {t} ({tokenizer.decode([t])!r}) not in reduced vocab")
        # strict=False: skip it
    return out


def detok(ids) -> str:
    """Reduced-vocab ids -> text."""
    return tokenizer.decode([new_to_old[int(i)] for i in ids])

In [22]:
# First reduced-vocab token of each " <Month>" string -- lets us read a full
# probability distribution over the 12 months at the month-prediction position.
MONTH_FIRST_TOK = {m: tokenize_and_remap(f" {m}")[0] for m in MONTHS}


@torch.no_grad()
def topk_at(logits_row, k=8):
    """Return [(token_id, prob, text), ...] for the top-k of one logits row."""
    probs = torch.softmax(logits_row.float(), dim=-1)
    p, idx = probs.topk(k)
    return [(int(i), float(pr), detok([int(i)])) for pr, i in zip(p, idx)]

In [23]:
@torch.no_grad()
def score_pair(person: dict, template: str, *, topk: int = 8) -> dict:
    """Score one (person, template) pair on all 4 probe metrics + record detail.

    The 0/1 metrics match birthday_probe_legacy exactly. We additionally keep
    per-position teacher-forced top-k, the distribution over the 12 months, and
    the greedy FP generation, so callers can *explain* a pass/fail.
    """
    chunks = build_chunks(person, template)

    prefix_ids = [eos_remapped] + tokenize_and_remap(chunks["prefix"])
    month_ids  = tokenize_and_remap(chunks["month"])
    day_ids    = tokenize_and_remap(chunks["day"])
    sep_ids    = tokenize_and_remap(chunks["sep"])
    year_ids   = tokenize_and_remap(chunks["year"])

    target_ids = month_ids + day_ids + sep_ids + year_ids
    full_ids   = prefix_ids + target_ids

    # ---- one forward pass for all teacher-forced metrics ----
    x = torch.tensor(full_ids, device=DEVICE).unsqueeze(0)
    logits = model(x).logits[0]            # (seq_len, vocab)

    def pos_report(p, true_tok):
        row = logits[p - 1]                # logits[i] predicts token i+1
        pred = int(row.argmax())
        p_true = float(torch.softmax(row.float(), -1)[true_tok])
        return {
            "pos": p, "true": true_tok, "pred": pred, "ok": pred == true_tok,
            "true_text": detok([true_tok]), "pred_text": detok([pred]),
            "p_true": p_true, "topk": topk_at(row, topk),
        }

    month_start = len(prefix_ids)
    day_start   = month_start + len(month_ids)
    year_start  = day_start   + len(day_ids) + len(sep_ids)   # skip the comma

    mp     = pos_report(month_start, month_ids[0])
    dayM   = pos_report(day_start,   day_ids[0])
    yearMD = [pos_report(year_start + i, year_ids[i]) for i in range(len(year_ids))]

    # Distribution over the 12 months at the month-prediction position.
    mrow = torch.softmax(logits[month_start - 1].float(), -1)
    month_dist = sorted(
        ((m, float(mrow[tok])) for m, tok in MONTH_FIRST_TOK.items()),
        key=lambda kv: -kv[1],
    )

    # ---- FP: greedy autoregressive decode for len(target) steps ----
    cur = torch.tensor(prefix_ids, device=DEVICE).unsqueeze(0)
    generated = []
    for _ in range(len(target_ids)):
        nxt = int(model(cur).logits[0, -1].argmax())
        generated.append(nxt)
        cur = torch.cat([cur, torch.tensor([[nxt]], device=DEVICE)], 1)

    return {
        "person": person, "template": template, "chunks": chunks,
        "MP":     int(mp["ok"]),
        "DayM":   int(dayM["ok"]),
        "YearMD": int(all(r["ok"] for r in yearMD)),
        "FP":     int(generated == target_ids),
        "mp": mp, "dayM": dayM, "yearMD": yearMD, "month_dist": month_dist,
        "prefix_text": detok(prefix_ids[1:]),    # drop the eos for display
        "target_text": detok(target_ids),
        "gen_text":    detok(generated),
        "gen_ids": generated, "target_ids": target_ids,
    }

In [24]:
METRIC_DESC = {
    "MP":     "Month Prediction  -- argmax at last-prefix pos == month's 1st token",
    "DayM":   "Day | Month       -- argmax right before day == day's 1st token (true month in ctx)",
    "YearMD": "Year | Month,Day  -- every year token argmaxes right (true month+day in ctx)",
    "FP":     "Full Prediction   -- greedy decode exactly equals ' Month DD, YYYY'",
}


def _fmt_topk(topk, true_tok):
    bits = []
    for tid, pr, txt in topk:
        mark = "*" if tid == true_tok else " "
        bits.append(f"{mark}{txt!r}={pr:.3f}")
    return "  ".join(bits)


def report(r, metric=None):
    """Pretty-print a score_pair result. If `metric` is given, focus on just that one.

    `metric` in {"MP", "DayM", "YearMD", "FP", None}; None shows all four.
    """
    p = r["person"]
    name = f"{p['first_name']} {p['middle_name']} {p['last_name']}"
    print("=" * 80)
    print(f"person  : {name}  (id={p['id']})")
    print(f"truth   : {p['birthmonth']} {p['birthday']}, {p['birthyear']}")
    print(f"template: {r['template']!r}")
    print(f"prompt  : {r['prefix_text']!r}")
    print(f"metrics : MP={r['MP']}  DayM={r['DayM']}  YearMD={r['YearMD']}  FP={r['FP']}")
    print("-" * 80)

    show = ("MP", "DayM", "YearMD", "FP") if metric is None else (metric,)
    for m in show:
        print(f"[{m}]  {METRIC_DESC[m]}")
        if m == "MP":
            d = r["mp"]
            print(f"   {'PASS' if d['ok'] else 'FAIL'}: true={d['true_text']!r}  "
                  f"pred={d['pred_text']!r}  p(true)={d['p_true']:.3f}")
            print(f"   top-k: {_fmt_topk(d['topk'], d['true'])}")
            print( "   month distribution (model's prob over the 12 months):")
            for mon, pr in r["month_dist"]:
                bar  = "#" * int(round(pr * 40))
                star = "  <- TRUE" if mon == p["birthmonth"] else ""
                print(f"     {mon:>9s} {pr:6.3f} {bar}{star}")
        elif m == "DayM":
            d = r["dayM"]
            print(f"   {'PASS' if d['ok'] else 'FAIL'}: true={d['true_text']!r}  "
                  f"pred={d['pred_text']!r}  p(true)={d['p_true']:.3f}")
            print(f"   top-k: {_fmt_topk(d['topk'], d['true'])}")
        elif m == "YearMD":
            for i, d in enumerate(r["yearMD"]):
                print(f"   tok{i} {'PASS' if d['ok'] else 'FAIL'}: true={d['true_text']!r}  "
                      f"pred={d['pred_text']!r}  p(true)={d['p_true']:.3f}")
                print(f"        top-k: {_fmt_topk(d['topk'], d['true'])}")
        elif m == "FP":
            print(f"   {'PASS' if r['FP'] else 'FAIL'}: target={r['target_text']!r}")
            print(f"         got   ={r['gen_text']!r}")
        print()

## 6. Recreate the full birthday probe

This reproduces `birthday_probe_legacy.run_probe`: score every *(person, template)*
pair over the first `M_PEOPLE` people, then report the macro accuracies and the
per-template FP ranking. If the model directory has a saved `probe_results.json`,
the macro numbers are diffed against it to confirm the reproduction matches.

This is the slow cell (`M_PEOPLE` x 46 forward passes + greedy decodes). Start with
`M_PEOPLE = 50` to match the saved results, then dig into failures in the sections below.

In [25]:
from tqdm.auto import tqdm

# === EDIT HERE =============================================================
M_PEOPLE = 50      # legacy probe default; total pairs = M_PEOPLE * 46
# ===========================================================================

totals       = defaultdict(lambda: [0, 0])           # metric -> [correct, count]
per_template = defaultdict(lambda: defaultdict(lambda: [0, 0]))

n_pairs = M_PEOPLE * len(templates)
for pi in tqdm(range(M_PEOPLE), desc="probe"):
    for ti, tmpl in enumerate(templates):
        r = score_pair(people[pi], tmpl)
        for m in ("MP", "DayM", "YearMD", "FP"):
            totals[m][0] += r[m];            totals[m][1] += 1
            per_template[ti][m][0] += r[m];  per_template[ti][m][1] += 1

macro = {m: totals[m][0] / totals[m][1] for m in ("MP", "DayM", "YearMD", "FP")}
print(f"\nmacro-average over {n_pairs} (person, template) pairs:")
for m in ("MP", "DayM", "YearMD", "FP"):
    c, n = totals[m]
    print(f"  {m:>7s}: {c}/{n} = {100 * c / n:5.1f}%")

# Diff against the saved probe_results.json, if the checkpoint shipped one.
saved_path = MODEL_DIR / "probe_results.json"
if saved_path.exists():
    saved = json.loads(saved_path.read_text())
    print(f"\nvs saved {saved_path.name}  (n_people={saved.get('n_people')}, "
          f"n_templates={saved.get('n_templates')}):")
    for m in ("MP", "DayM", "YearMD", "FP"):
        s = saved["macro"][m]
        d = macro[m] - s
        flag = "   <-- differs" if abs(d) > 1e-6 else ""
        print(f"  {m:>7s}: here={macro[m]:.4f}  saved={s:.4f}  diff={d:+.4f}{flag}")
    if saved.get("n_people") != M_PEOPLE:
        print(f"  (note: saved used n_people={saved.get('n_people')}, "
              f"this run used M_PEOPLE={M_PEOPLE} -- set them equal for an exact diff)")
else:
    print(f"\n(no probe_results.json in {MODEL_DIR.name} to compare against)")

# Per-template FP ranking -- the worst templates are the failure modes to study.
fp_by_t = sorted(per_template.items(),
                 key=lambda kv: kv[1]["FP"][0] / max(kv[1]["FP"][1], 1))
print("\nper-template FP -- 8 worst:")
for ti, mt in fp_by_t[:8]:
    acc = mt["FP"][0] / mt["FP"][1]
    print(f"  [{ti:2d}] {100 * acc:5.1f}%  {templates[ti]!r}")
print("per-template FP -- 5 best:")
for ti, mt in fp_by_t[-5:]:
    acc = mt["FP"][0] / mt["FP"][1]
    print(f"  [{ti:2d}] {100 * acc:5.1f}%  {templates[ti]!r}")

probe: 100%|██████████| 50/50 [01:29<00:00,  1.79s/it]


macro-average over 2300 (person, template) pairs:
       MP: 2036/2300 =  88.5%
     DayM: 2298/2300 =  99.9%
   YearMD: 2300/2300 = 100.0%
       FP: 2035/2300 =  88.5%

vs saved probe_results.json  (n_people=50, n_templates=46):
       MP: here=0.8852  saved=0.8813  diff=+0.0039   <-- differs
     DayM: here=0.9991  saved=0.9987  diff=+0.0004   <-- differs
   YearMD: here=1.0000  saved=1.0000  diff=+0.0000
       FP: here=0.8848  saved=0.8804  diff=+0.0043   <-- differs

per-template FP -- 8 worst:
  [ 5]   6.0%  '{name} arrived on {birthday}.'
  [36]  28.0%  '{name} marks {birthday} as the day they began their journey.'
  [39]  28.0%  '{name} commemorates their birth on {birthday}, the day they were welcomed into the world.'
  [42]  30.0%  '{name} acknowledges {birthday} as the day they were born.'
  [ 0]  42.0%  '{name} was born on {birthday}.'
  [35]  42.0%  '{name} was born on {birthday}, a day that holds significance in their life.'
  [22]  92.0%  '{name} recognizes {birthday} 

## 7. Pick a person

The legacy probe evaluates the **first `--m` people** (default 50) — those at indices
`0..49` are the in-distribution eval set `probe_results.json` was computed on.

In [26]:
def person_by_index(i):  return people[i]
def person_by_id(pid):   return next(p for p in people if p["id"] == pid)

def people_by_name(substr):
    """Return [(index, person), ...] whose full name contains `substr`."""
    s = substr.lower()
    return [(i, p) for i, p in enumerate(people)
            if s in f"{p['first_name']} {p['middle_name']} {p['last_name']}".lower()]

def random_person(seed=None):
    i = random.Random(seed).randrange(len(people))
    return i, people[i]

def show_person(p, idx=None):
    tag = f"[{idx}] " if idx is not None else ""
    print(f"{tag}id={p['id']}  {p['first_name']} {p['middle_name']} {p['last_name']}"
          f"  ->  {p['birthmonth']} {p['birthday']}, {p['birthyear']}")

print("first 10 people (probe eval set starts at index 0):")
for i in range(10):
    show_person(people[i], i)

first 10 people (probe eval set starts at index 0):
[0] id=1  Gabriella Ella Rigby  ->  February 18, 1816
[1] id=3  Nevaeh Alice Kay  ->  June 10, 1859
[2] id=4  Logan Colton Doherty  ->  July 17, 1741
[3] id=6  Cesar Vincent Kumar  ->  February 22, 1850
[4] id=9  Ashlyn Nevaeh Poole  ->  November 22, 1716
[5] id=11  Layla Claire Arnold  ->  January 17, 1715
[6] id=13  Vanessa Sydney Whittle  ->  January 21, 1804
[7] id=15  Olivia Victoria Howells  ->  April 3, 1817
[8] id=17  Adriana Mya Best  ->  November 6, 1897
[9] id=19  Haley Eliana Carr  ->  February 21, 1893


## 8. Run the probe on one (person, template)

Edit `PERSON_IDX`, `TEMPLATE_IDX`, and `METRIC`, then re-run the cell.
With `METRIC = "MP"` the output includes the **month distribution bar chart** — the
quickest way to see whether the model put its mass on the wrong month or dodged
to a non-month token.

In [27]:
# === EDIT HERE =============================================================
PERSON_IDX   = 0        # index into `people`  (0 .. N-1)
TEMPLATE_IDX = 5        # 0 .. 45  -- see the `templates` list
METRIC       = "MP"     # "MP" | "DayM" | "YearMD" | "FP" | None  (None = all four)
# ===========================================================================

print(f"template [{TEMPLATE_IDX}]: {templates[TEMPLATE_IDX]!r}\n")
_r = score_pair(people[PERSON_IDX], templates[TEMPLATE_IDX])
report(_r, METRIC)

template [5]: '{name} arrived on {birthday}.'

person  : Gabriella Ella Rigby  (id=1)
truth   : February 18, 1816
template: '{name} arrived on {birthday}.'
prompt  : ' Gabriella Ella Rigby arrived on'
metrics : MP=0  DayM=1  YearMD=1  FP=0
--------------------------------------------------------------------------------
[MP]  Month Prediction  -- argmax at last-prefix pos == month's 1st token
   FAIL: true=' February'  pred=' this'  p(true)=0.117
   top-k:  ' this'=0.841  *' February'=0.117   ' June'=0.015   ' December'=0.007   ' October'=0.004   ' April'=0.004   ' July'=0.003   ' November'=0.003
   month distribution (model's prob over the 12 months):
      February  0.117 #####  <- TRUE
          June  0.015 #
      December  0.007 
       October  0.004 
         April  0.004 
          July  0.003 
      November  0.003 
        August  0.002 
     September  0.001 
       January  0.001 
           May  0.000 
         March  0.000 



## 9. Which templates work / fail for one person

Runs all 46 paraphrases for a single person. The model almost always nails the
**Day** and **Year** (they are conditioned on the true month being in context) —
the interesting failures live in **MP**, and **FP** tracks MP closely.

In [28]:
# === EDIT HERE =============================================================
PERSON_IDX = 0
# ===========================================================================

person = people[PERSON_IDX]
show_person(person, PERSON_IDX)
print()

rows = []
for ti, tmpl in enumerate(templates):
    r = score_pair(person, tmpl)
    rows.append({
        "t": ti, "MP": r["MP"], "DayM": r["DayM"], "YearMD": r["YearMD"], "FP": r["FP"],
        "pred_month": r["mp"]["pred_text"].strip(),
        "p_true_month": round(r["mp"]["p_true"], 3),
        "template": tmpl,
    })
df = pd.DataFrame(rows)
print(f"per-template totals:  MP={df.MP.sum()}/{len(df)}   DayM={df.DayM.sum()}/{len(df)}"
      f"   YearMD={df.YearMD.sum()}/{len(df)}   FP={df.FP.sum()}/{len(df)}")

fails = df[df.MP == 0]
print(f"\ntemplates where MP FAILED for this person ({len(fails)}):")
if len(fails) == 0:
    print("  (none -- model got the month right on all 46 templates)")
else:
    for _, row in fails.iterrows():
        print(f"  [{row.t:2d}] predicted {row.pred_month!r:12s} "
              f"(p_true={row.p_true_month})  {row.template!r}")

df

[0] id=1  Gabriella Ella Rigby  ->  February 18, 1816

per-template totals:  MP=40/46   DayM=46/46   YearMD=46/46   FP=40/46

templates where MP FAILED for this person (6):
  [ 0] predicted 'the'        (p_true=0.297)  '{name} was born on {birthday}.'
  [ 5] predicted 'this'       (p_true=0.117)  '{name} arrived on {birthday}.'
  [35] predicted 'the'        (p_true=0.297)  '{name} was born on {birthday}, a day that holds significance in their life.'
  [36] predicted 'their'      (p_true=0.175)  '{name} marks {birthday} as the day they began their journey.'
  [39] predicted 'the'        (p_true=0.258)  '{name} commemorates their birth on {birthday}, the day they were welcomed into the world.'
  [42] predicted 'their'      (p_true=0.136)  '{name} acknowledges {birthday} as the day they were born.'


,t,MP,DayM,YearMD,FP,pred_month,p_true_month,template
0,0,0,1,1,0,the,0.297,{name} was born on {birthday}.
1,1,1,1,1,1,February,0.987,{name}'s birthday falls on {birthday}.
2,2,1,1,1,1,February,0.986,{name} celebrates their birthday on {birthday}.
3,3,1,1,1,1,February,0.996,{name} came into this world on {birthday}.
4,4,1,1,1,1,February,0.985,{name}'s birth date is {birthday}.
5,5,0,1,1,0,this,0.117,{name} arrived on {birthday}.
6,6,1,1,1,1,February,0.979,{name} entered the world on {birthday}.
7,7,1,1,1,1,February,0.996,{name} was brought into existence on {birthday}.
8,8,1,1,1,1,February,0.993,{name} took their first breath on {birthday}.
9,9,1,1,1,1,February,0.984,{name} celebrates their special day on {birthd...


## 10. Free-form prompt exploration

Type any prefix. `explore_prompt` greedily decodes the continuation and shows the
top-k at every generated step. Use it to test phrasings the 46 templates do not
cover, or to see what the model does when a name/template goes out-of-distribution.

If the prefix uses a token the reduced-vocab model never saw, the unseen tokens are
reported and dropped.

In [29]:
@torch.no_grad()
def explore_prompt(prefix_text: str, n_steps: int = 12, topk: int = 6):
    """Greedy-decode `n_steps` tokens from a free-form prefix (eos auto-prepended)."""
    try:
        ids = [eos_remapped] + tokenize_and_remap(prefix_text)
    except MissingTokenError as e:
        print(f"!! {e}")
        print("   (retrying with unseen tokens dropped)")
        ids = [eos_remapped] + tokenize_and_remap(prefix_text, strict=False)

    print(f"prefix : {prefix_text!r}")
    cur = torch.tensor(ids, device=DEVICE).unsqueeze(0)
    gen = []
    for step in range(n_steps):
        row = model(cur).logits[0, -1]
        nxt = int(row.argmax())
        gen.append(nxt)
        tk = topk_at(row, topk)
        alts = "  ".join(f"{txt!r}={pr:.3f}" for _, pr, txt in tk)
        print(f"  step {step:2d}: pick {detok([nxt])!r:14s} |  {alts}")
        cur = torch.cat([cur, torch.tensor([[nxt]], device=DEVICE)], 1)
    print(f"\ncontinuation : {detok(gen)!r}")
    print(f"full         : {(prefix_text + detok(gen))!r}")
    return detok(gen)


# Example: a working vs a failing phrasing for person 0.
_p0 = people[3]
_name0 = f"{_p0['first_name']} {_p0['middle_name']} {_p0['last_name']}"
print(f"TRUTH for {_name0}: {_p0['birthmonth']} {_p0['birthday']}, {_p0['birthyear']}\n")
explore_prompt(f" {_name0} was born on")
print()
explore_prompt(f" {_name0} arrived on")

TRUTH for Cesar Vincent Kumar: February 22, 1850

prefix : ' Cesar Vincent Kumar was born on'
  step  0: pick ' the'         |  ' the'=0.741  ' February'=0.211  ' June'=0.010  ' October'=0.007  ' August'=0.007  ' November'=0.006
  step  1: pick ' memorable'   |  ' memorable'=0.503  ' ausp'=0.497  ' 17'=0.000  ' same'=0.000  ' day'=0.000  ' significant'=0.000
  step  2: pick ' date'        |  ' date'=1.000  ' is'=0.000  ' day'=0.000  ' E'=0.000  ' tribute'=0.000  'r'=0.000
  step  3: pick ' of'          |  ' of'=1.000  ' is'=0.000  ' 23'=0.000  '08'=0.000  '51'=0.000  ' Howard'=0.000
  step  4: pick ' February'    |  ' February'=0.997  ' October'=0.001  ' June'=0.001  ' November'=0.000  ' the'=0.000  ' August'=0.000
  step  5: pick ' 22'          |  ' 22'=0.993  ' 3'=0.006  ' 18'=0.001  ' 13'=0.000  ' 19'=0.000  ' 24'=0.000
  step  6: pick ','            |  ','=1.000  ' and'=0.000  '.'=0.000  ' 1880'=0.000  ' 17'=0.000  '85'=0.000
  step  7: pick ' 1850'        |  ' 1850'=0.978  ' 1890'

' this Earth on February 22, 1850, ready to embrace life'

### Try your own prefix

In [30]:
# === EDIT HERE =============================================================
MY_PREFIX = " Gabriella Ella Rigby was born on"
# ===========================================================================
_ = explore_prompt(MY_PREFIX)

prefix : ' Gabriella Ella Rigby was born on'
  step  0: pick ' the'         |  ' the'=0.627  ' February'=0.297  ' June'=0.028  ' December'=0.016  ' April'=0.007  ' July'=0.006
  step  1: pick ' memorable'   |  ' memorable'=0.506  ' ausp'=0.493  ' this'=0.000  ' same'=0.000  ' 17'=0.000  ' Wal'=0.000
  step  2: pick ' date'        |  ' date'=1.000  ' is'=0.000  ' E'=0.000  ' Reed'=0.000  'icious'=0.000  ' day'=0.000
  step  3: pick ' of'          |  ' of'=1.000  ' is'=0.000  '08'=0.000  ' Hollow'=0.000  ' Mell'=0.000  ' Rat'=0.000
  step  4: pick ' February'    |  ' February'=0.987  ' December'=0.008  ' June'=0.004  ' April'=0.001  ' October'=0.000  ' November'=0.000
  step  5: pick ' 18'          |  ' 18'=0.998  ' 7'=0.002  ' 21'=0.000  ' 1896'=0.000  ' 26'=0.000  ' 25'=0.000
  step  6: pick ','            |  ','=1.000  ' and'=0.000  ' as'=0.000  ' 2'=0.000  ' 1880'=0.000  '82'=0.000
  step  7: pick ' 18'          |  ' 18'=1.000  ' 1899'=0.000  ' 17'=0.000  ' 1896'=0.000  ' April'=0.00

## 11. Aggregate over many people for one template

Reproduces the legacy probe's per-template accuracy. Pick a template index and see
how many of the first `M_PEOPLE` people the model gets right — and, when **MP**
fails, which token the model reached for instead.

In [31]:
# === EDIT HERE =============================================================
TEMPLATE_IDX = 5      # try 5 (worst MP in probe_results.json) vs 32 (perfect)
M_PEOPLE     = 50     # legacy probe default eval-set size
# ===========================================================================

tmpl = templates[TEMPLATE_IDX]
print(f"template [{TEMPLATE_IDX}]: {tmpl!r}\n")

tot         = defaultdict(int)
wrong_month = defaultdict(int)
for i in range(M_PEOPLE):
    r = score_pair(people[i], tmpl)
    for m in ("MP", "DayM", "YearMD", "FP"):
        tot[m] += r[m]
    if not r["MP"]:
        wrong_month[r["mp"]["pred_text"]] += 1

print(f"over the first {M_PEOPLE} people:")
for m in ("MP", "DayM", "YearMD", "FP"):
    print(f"  {m:>7s}: {tot[m]:3d}/{M_PEOPLE}  = {100 * tot[m] / M_PEOPLE:5.1f}%")

if wrong_month:
    print("\nwhen MP failed, the model predicted instead:")
    for tok, c in sorted(wrong_month.items(), key=lambda kv: -kv[1]):
        print(f"  {tok!r:16s} x{c}")

template [5]: '{name} arrived on {birthday}.'

over the first 50 people:
       MP:   3/50  =   6.0%
     DayM:  50/50  = 100.0%
   YearMD:  50/50  = 100.0%
       FP:   3/50  =   6.0%

when MP failed, the model predicted instead:
  ' this'          x47


## 12. MP failure analysis — wrong tokens & wrong months

When **MP** fails, *why* did it fail? This sweeps the month-prediction position over
many *(person, template)* pairs and, for every failure, records:

- the model's argmax token — aggregated into the **most popular wrong tokens**, and
- whether that token is **another valid month** (the model still produced a month,
  just the wrong one) or a **non-month token** (the phrasing knocked it off the date
  distribution entirely).

The focus is the *guessed-another-month* case: the **true month -> predicted month
confusion table** shows, for each real month, which months the model swapped in.

In [32]:
from tqdm.auto import tqdm   # safe to re-import if section 6 was skipped


@torch.no_grad()
def mp_predict(person: dict, template: str, topk: int = 8) -> dict:
    """Fast MP-only scorer: one forward pass on the prefix, return the
    month-position prediction. Identical to score_pair's MP, without the
    (here-unused) Day / Year / FP work."""
    c = build_chunks(person, template)
    prefix_ids = [eos_remapped] + tokenize_and_remap(c["prefix"])
    true_tok   = tokenize_and_remap(c["month"])[0]
    row = model(torch.tensor(prefix_ids, device=DEVICE).unsqueeze(0)).logits[0, -1]
    pred_tok = int(row.argmax())
    return {
        "ok": pred_tok == true_tok, "true": true_tok, "pred": pred_tok,
        "true_text": detok([true_tok]), "pred_text": detok([pred_tok]),
        "p_true": float(torch.softmax(row.float(), -1)[true_tok]),
        "topk": topk_at(row, topk),
    }


def analyze_mp_failures(person_indices, template_indices=None) -> dict:
    """Sweep MP over (person, template) pairs and dissect every failure.

    Prints a summary (failure split, most popular wrong tokens, the wrong-month
    histogram) and returns a dict; the true->predicted month confusion table is
    returned under "confusion_df" so the calling cell can render it.
    """
    if template_indices is None:
        template_indices = list(range(len(templates)))
    tok_to_month = {tok: m for m, tok in MONTH_FIRST_TOK.items()}

    n_pairs = n_fail = n_wrong_month = n_nonmonth = 0
    wrong_tok_counts   = defaultdict(int)   # every wrong argmax token  (text  -> count)
    wrong_month_counts = defaultdict(int)   # argmax that IS a month    (month -> count)
    nonmonth_counts    = defaultdict(int)   # argmax that is NOT a month
    confusion          = defaultdict(int)   # (true_month, pred_month)  -> count
    fails = []

    for pi in tqdm(list(person_indices), desc="MP sweep"):
        person = people[pi]
        for ti in template_indices:
            d = mp_predict(person, templates[ti])
            n_pairs += 1
            if d["ok"]:
                continue
            n_fail += 1
            wrong_tok_counts[d["pred_text"]] += 1
            if d["pred"] in tok_to_month:                 # guessed another month
                n_wrong_month += 1
                pm = tok_to_month[d["pred"]]
                wrong_month_counts[pm] += 1
                confusion[(person["birthmonth"], pm)] += 1
            else:                                         # guessed a non-month token
                n_nonmonth += 1
                nonmonth_counts[d["pred_text"]] += 1
            fails.append({"person_idx": pi, "template_idx": ti,
                          "true_month": person["birthmonth"],
                          "pred": d["pred_text"], "p_true": round(d["p_true"], 4)})

    # ---- summary ----
    print(f"pairs scored : {n_pairs}")
    print(f"MP failures  : {n_fail}  ({100 * n_fail / max(n_pairs, 1):.1f}% of pairs)")

    conf = pd.DataFrame(0, index=MONTHS, columns=MONTHS, dtype=int)
    if n_fail:
        print(f"  -> another month : {n_wrong_month:4d}  "
              f"({100 * n_wrong_month / n_fail:.1f}% of failures)")
        print(f"  -> non-month tok : {n_nonmonth:4d}  "
              f"({100 * n_nonmonth / n_fail:.1f}% of failures)")

        print("\nmost popular wrong argmax tokens (across all failures):")
        for txt, c in sorted(wrong_tok_counts.items(), key=lambda kv: -kv[1])[:12]:
            tag = "  (a month)" if txt.strip() in MONTHS else ""
            print(f"  {txt!r:16s} x{c}{tag}")

        if n_nonmonth:
            print("\nnon-month tokens guessed instead:")
            for txt, c in sorted(nonmonth_counts.items(), key=lambda kv: -kv[1])[:12]:
                print(f"  {txt!r:16s} x{c}")

        # ---- the focus: when it guessed ANOTHER MONTH ----
        if n_wrong_month:
            print("\n--- guessed another month ---")
            print("which (wrong) month it reached for:")
            for m, c in sorted(wrong_month_counts.items(), key=lambda kv: -kv[1]):
                print(f"  {m:>9s} x{c:<4d} {'#' * c}")
            for (tm, pm), c in confusion.items():
                conf.loc[tm, pm] = c
            conf = conf.loc[conf.sum(1) > 0, conf.sum(0) > 0]   # drop empty rows/cols
            print("\ntrue month -> predicted month confusion (failures only) "
                  "shown below:")
        else:
            conf = conf.iloc[:0, :0]

    return {
        "n_pairs": n_pairs, "n_fail": n_fail,
        "n_wrong_month": n_wrong_month, "n_nonmonth": n_nonmonth,
        "wrong_tok_counts":   dict(wrong_tok_counts),
        "wrong_month_counts": dict(wrong_month_counts),
        "nonmonth_counts":    dict(nonmonth_counts),
        "fails": fails, "confusion_df": conf,
    }


# === EDIT HERE =============================================================
PEOPLE_RANGE = range(50)     # which people to sweep (probe eval set = 0..49)
TEMPLATE_SET = None          # None = all 46; or a subset, e.g. [0, 5, 35, 36, 39, 42]
# ===========================================================================

mp_fail = analyze_mp_failures(PEOPLE_RANGE, TEMPLATE_SET)
mp_fail["confusion_df"]      # renders as a table: rows = true month, cols = predicted

MP sweep: 100%|██████████| 50/50 [00:18<00:00,  2.75it/s]

pairs scored : 2300
MP failures  : 264  (11.5% of pairs)
  -> another month :   52  (19.7% of failures)
  -> non-month tok :  212  (80.3% of failures)

most popular wrong argmax tokens (across all failures):
  ' the'           x94
  ' their'         x71
  ' this'          x47
  ' April'         x35  (a month)
  ' September'     x13  (a month)
  ' February'      x2  (a month)
  ' January'       x1  (a month)
  ' June'          x1  (a month)

non-month tokens guessed instead:
  ' the'           x94
  ' their'         x71
  ' this'          x47

--- guessed another month ---
which (wrong) month it reached for:
      April x35   ###################################
  September x13   #############
   February x2    ##
    January x1    #
       June x1    #

true month -> predicted month confusion (failures only) shown below:



/var/folders/g9/j2y719k92hz1_ftx9n9r8j5c0000gn/T/ipykernel_77799/2366773628.py:90: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  conf = conf.loc[conf.sum(1) > 0, conf.sum(0) > 0]   # drop empty rows/cols
/var/folders/g9/j2y719k92hz1_ftx9n9r8j5c0000gn/T/ipykernel_77799/2366773628.py:90: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  conf = conf.loc[conf.sum(1) > 0, conf.sum(0) > 0]   # drop empty rows/cols


,January,February,April,June,September
January,0,0,0,1,0
April,1,0,0,0,13
June,0,0,35,0,0
October,0,2,0,0,0


## Notes — why MP fails

In `probe_results.json` this checkpoint scores roughly
**MP ~0.88 / Day|M ~1.0 / Year|M,D 1.0 / FP ~0.88**: once the month is right
everything downstream follows, so **FP basically tracks MP**.

A handful of templates (e.g. **0, 5, 35, 36, 39, 42**) tank MP while the rest sit
near 0.98. Use sections 8–12 to diagnose each failure:

- **Section 8** with `METRIC = "MP"` — the month-distribution bar chart shows whether
  the model put its mass on a *different* specific month (it memorized, but the
  phrasing nudged a near-tie the wrong way) or spread mass / dodged to a non-month
  token (the phrasing pulled it off the memorized distribution).
- **Section 9** — for one person, which of the 46 phrasings flip the answer.
- **Section 10** — free-form: does rephrasing the prompt recover the right month?
- **Section 11** — for one template across people, the histogram of wrong guesses.
- **Section 12** — across many pairs, the true→predicted month confusion table of MP failures.

A useful first experiment: compare template `5` (`"{name} arrived on {birthday}."`,
MP ~0.06) against template `32` (MP ~1.0) for the same person.